In [1]:
import pandas as pd
import numpy as np

In [2]:
files = [
    "/content/IT Support Ticket Data.csv",
    "/content/customer_support_tickets_200k.csv",
    "/content/customer_support_tickets_200k.csv"
]

In [3]:
dfs = []
for f in files:
  temp_df = pd.read_csv(f)
  if "body" in temp_df.columns:
    temp_df = temp_df["body"]
  elif "issue_description" in temp_df.columns:
    temp_df = temp_df["issue_description"]
  elif "Body" in temp_df.columns:
    temp_df = temp_df["Body"]
  dfs.append(temp_df)

In [4]:
df = pd.concat(dfs, ignore_index = True)

In [5]:
df = df.to_frame(name = "description")

In [6]:
df.head(5)

,description
0,"Dear Customer Support Team,I am writing to rep..."
1,"Dear Customer Support Team,I hope this message..."
2,"Dear Customer Support Team,I hope this message..."
3,"Dear Support Team,I hope this message reaches ..."
4,"Dear Customer Support,I hope this message reac..."


In [7]:
df = df.dropna()

In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [9]:
def clean_text(text, keep_punctuation=True, lemmatize=False, remove_stopwords=False):
    """
    Enhanced text cleaning with configurable options
    """
    original_text = text

    text = text.lower()

    text = re.sub(r"^(dear|hi|hello|hey)\s+.*?,", "", text, flags=re.IGNORECASE)

    polite_phrases = [
        r"i hope this message finds you well",
        r"i hope this message reaches you",
        r"i hope you are doing well",
        r"i am reaching out to",
        r"just writing to",
        r"i wanted to",
        r"i would like to",
        r"i am writing to (report|inform|ask|request)",
        r"please (find|see) attached",
        r"attached please find"
    ]

    for phrase in polite_phrases:
        text = re.sub(phrase, "", text, flags=re.IGNORECASE)

    signoffs = [
        r"thank you.*$", r"thanks.*$", r"regards.*$",
        r"best regards.*$", r"sincerely.*$", r"cheers.*$",
        r"appreciate your (help|assistance|time).*$",
        r"looking forward.*$"
    ]

    for signoff in signoffs:
        text = re.sub(signoff, "", text, flags=re.IGNORECASE)

    text = re.sub(r"\b\w+@\w+\.\w+\b", "", text)  # emails
    text = re.sub(r"\b\d{10,}\b", "", text)  # phone numbers
    text = re.sub(r"ticket\s*id:?\s*\w+", "", text, flags=re.IGNORECASE)  # ticket IDs

    if keep_punctuation:
        text = re.sub(r"[^\w\s\.\?\!]", "", text)
    else:
        text = re.sub(r"[^\w\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    if remove_stopwords or lemmatize:
        tokens = text.split()

        if remove_stopwords:
            stop_words = set(stopwords.words('english'))
            domain_words = {'not', 'no', 'but', 'however', 'issue', 'problem', 'error'}
            stop_words = stop_words - domain_words
            tokens = [word for word in tokens if word not in stop_words]

        if lemmatize:
            lemmatizer = WordNetLemmatizer()
            tokens = [lemmatizer.lemmatize(word) for word in tokens]

        text = ' '.join(tokens)

    if not text or len(text.split()) < 2:
        text = original_text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
df["cleaned_text"] = df["description"].apply(clean_text)

In [11]:
df = df.drop_duplicates(subset=['cleaned_text'], keep='first')

print(f"Original rows: {len(df)}")
print(f"After dedup on cleaned_text: {len(df)}")

Original rows: 25047
After dedup on cleaned_text: 25047


In [12]:
df["cleaned_text"]

,cleaned_text
0,a significant problem with the centralized acc...
1,well. request detailed information about the c...
2,. request clarification about the billing and ...
3,well. ask about the compatibility of your prod...
4,in good health. i am eager to learn more about...
...,...
29656,there seems to be a discrepancy in my billing ...
29657,request a refund for the recent charge.
29658,twofactor authentication codes are not being d...
29660,i am unable to access my account after enterin...


In [13]:
%%capture
!pip install sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer
import torch

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [16]:
model_name = "all-mpnet-base-v2"

In [17]:
embedder = SentenceTransformer(model_name)
embedder.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [18]:
texts = df["cleaned_text"].tolist()

In [19]:
embeddings = embedder.encode(
    texts,
    batch_size = 64,
    show_progress_bar = True,
    normalize_embeddings = True
)

Batches:   0%|          | 0/392 [00:00<?, ?it/s]

In [20]:
%%capture
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [21]:
import cuml.cluster

In [22]:
import cuml

umap_model = cuml.UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine'
)
reduced_embeddings = umap_model.fit_transform(embeddings)

hdb = cuml.cluster.HDBSCAN(
    min_cluster_size=30,
    min_samples=1,
    metric='euclidean',
    cluster_selection_epsilon=0.1,
    cluster_selection_method='eom'
)
labels_gpu = hdb.fit_predict(reduced_embeddings)

In [23]:
n_clusters = len(np.unique(labels_gpu)) - (1 if -1 in labels_gpu else 0)
print(f"GPU HDBSCAN: {n_clusters} clusters")

GPU HDBSCAN: 162 clusters


In [24]:
unique, counts = np.unique(labels_gpu, return_counts=True)
n_clusters = len(unique) - (1 if -1 in unique else 0)
noise_count = counts[unique == -1][0] if -1 in unique else 0

print(f"Total clusters: {n_clusters}")
print(f"Noise / outliers: {noise_count} ({noise_count / len(labels_gpu) * 100:.1f}%)")
print(f"Largest 10 clusters sizes: {sorted(counts[unique != -1], reverse=True)[:10]}")
print(f"Small clusters (<20 tickets): {sum(counts < 20)}")

Total clusters: 162
Noise / outliers: 3269 (13.1%)
Largest 10 clusters sizes: [np.int64(2399), np.int64(1510), np.int64(1153), np.int64(825), np.int64(820), np.int64(546), np.int64(499), np.int64(489), np.int64(400), np.int64(382)]
Small clusters (<20 tickets): 0


In [25]:
import cuml
import cupy as cp

is_clustered = labels_gpu != -1
is_noise = labels_gpu == -1

knn = cuml.neighbors.KNeighborsClassifier(n_neighbors=1)
knn.fit(embeddings[is_clustered], labels_gpu[is_clustered])

noise_reassignments = knn.predict(embeddings[is_noise])

final_labels = labels_gpu.copy()
final_labels[is_noise] = noise_reassignments

noise_count = cp.sum(final_labels == -1)
print(f"Remaining noise: {noise_count}")

Remaining noise: 0


In [26]:
from sklearn.feature_extraction.text import CountVectorizer

def get_cluster_keywords(texts, labels, n_words=10):
    df = pd.DataFrame({'text': texts, 'label': labels})
    docs_per_class = df.groupby(['label'], as_index=False).agg({'text': ' '.join})

    count = CountVectorizer(stop_words='english').fit(docs_per_class.text)
    t = count.transform(docs_per_class.text).toarray()
    w = t.sum(axis=1)
    tf = np.divide(t.T, w)
    sum_t = t.sum(axis=0)
    idf = np.log(np.divide(len(docs_per_class), sum_t, out=np.zeros_like(sum_t, dtype=float), where=sum_t!=0))
    tf_idf = np.multiply(tf.T, idf)

    words = count.get_feature_names_out()
    top_n_words = {label: [words[i] for i in tf_idf[i_label].argsort()[-n_words:]]
                   for i_label, label in enumerate(docs_per_class.label)}
    return top_n_words

keywords = get_cluster_keywords(texts, final_labels)

In [27]:
keywords

{0: ['computing',
  'deployments',
  'costeffectiveness',
  'savings',
  'instances',
  'costeffective',
  'ec2',
  'unnecessary',
  'instance',
  'expenses'],
 1: ['reflects',
  'adjustment',
  'expenses',
  'incurred',
  'spike',
  'clarifying',
  'correspondence',
  'correction',
  'months',
  'justify'],
 2: ['jam',
  'wirelessly',
  'paper',
  'printing',
  'print',
  'mobile',
  'duplex',
  'canon',
  'pixma',
  'mg3620'],
 3: ['ink',
  'jams',
  'print',
  'paper',
  'epson',
  'et4760',
  'ecotank',
  'hp',
  'deskjet',
  '3755'],
 4: ['internally',
  'onsite',
  'dropouts',
  'technician',
  'weeks',
  'drops',
  'routers',
  'networking',
  'isr4331',
  'enterprise'],
 5: ['tablet',
  'repair',
  '7s',
  'responsiveness',
  'k780',
  'touch',
  'surface',
  'keyboard',
  'detachable',
  'touchscreen'],
 6: ['assistancesincerely',
  'capabilitiesthank',
  'screenshots',
  '2019',
  'cases',
  'enhances',
  'manipulation',
  'alteryxs',
  'gained',
  'alteryx'],
 7: ['disclosin

In [28]:
import cupy as cp
import pandas as pd
from sklearn.cluster import AgglomerativeClustering


unique_labels = sorted([l for l in np.unique(final_labels) if l != -1])
centroids = cp.array([embeddings[final_labels == l].mean(axis=0) for l in unique_labels])

n_meta_clusters = 30
merger = AgglomerativeClustering(n_clusters=n_meta_clusters, metric='cosine', linkage='average')
meta_labels = merger.fit_predict(centroids.get())

cluster_to_meta_map = {old_idx: meta_idx for old_idx, meta_idx in zip(unique_labels, meta_labels)}

df['cluster_id'] = final_labels
df['meta_cluster_id'] = df['cluster_id'].map(cluster_to_meta_map)

df['meta_label_name'] = df['meta_cluster_id'].apply(lambda x: f"Meta-Group {x}")

In [29]:
df.head(5)

,description,cleaned_text,cluster_id,meta_cluster_id,meta_label_name
0,"Dear Customer Support Team,I am writing to rep...",a significant problem with the centralized acc...,121,0,Meta-Group 0
1,"Dear Customer Support Team,I hope this message...",well. request detailed information about the c...,32,10,Meta-Group 10
2,"Dear Customer Support Team,I hope this message...",. request clarification about the billing and ...,69,2,Meta-Group 2
3,"Dear Support Team,I hope this message reaches ...",well. ask about the compatibility of your prod...,21,10,Meta-Group 10
4,"Dear Customer Support,I hope this message reac...",in good health. i am eager to learn more about...,82,21,Meta-Group 21


In [30]:
df['meta_label_name'].value_counts()

,count
meta_label_name,
Meta-Group 0,5182
Meta-Group 28,3321
Meta-Group 14,3143
Meta-Group 10,3108
Meta-Group 5,2407
Meta-Group 8,2157
Meta-Group 6,1724
Meta-Group 2,1568
Meta-Group 9,301


In [31]:
from collections import Counter

In [32]:
cluster_counts = Counter(labels_gpu)
sorted_clusters = sorted(cluster_counts.items(), key = lambda x: x[1], reverse = True)

In [33]:
cluster_counts = df["meta_cluster_id"].value_counts()
top_clusters = cluster_counts.index.sort_values()

for cid in top_clusters:
    if cid == -1: continue

    cluster_df = df[df["meta_cluster_id"] == cid]
    cluster_size = len(cluster_df)

    if cluster_size == 0:
        continue

    print(f"\n--- Meta-Group {cid} ({cluster_size} tickets) ---")

    samples = cluster_df["cleaned_text"].sample(
        min(5, cluster_size),
        random_state=42
    )

    for i, text in enumerate(samples, 1):
        preview = str(text).replace('\n', ' ').strip()
        print(f"  {i}. {preview[:180]}...")


--- Meta-Group 0 (5182 tickets) ---
  1. our saas system has encountered a freeze which has caused a loss of project data. this issue might be linked to the recent unifi software update or an hdmi cable problem. despite a...
  2. our marketing firm has encountered several failures during product integration likely due to software conflicts. we attempted to resolve the problem by rebooting systems and updati...
  3. have encountered sporadic access problems with the saas platform potentially due to recent server updates causing overload. initial diagnostics involved restarting servers and reve...
  4. noted a sharp decline in system performance during peak usage times. this might be due to a surge in user activity or inefficient queries. tried restarting the server and optimizin...
  5. i am contacting you to report an integration failure between monday.com and sap erp. it might be due to a mismatch in the api versions. i have already tried to resolve the issue by...

--- Meta-Group 1 

In [34]:
cluster_names = {
    'Meta-Group 0': 'Financial Data Analytics Support',
    'Meta-Group 1': 'System Outage & Service Disruption',
    'Meta-Group 2': 'General Technical Issues (Application & System Errors)',
    'Meta-Group 3': 'Documentation & Product Inquiries',
    'Meta-Group 4': 'Billing & Payment Issues',
    'Meta-Group 5': 'Third-Party Tools & Integration Requests',
    'Meta-Group 6': 'Service Downtime & Availability Issues',
    'Meta-Group 7': 'Application Functionality & Sync Issues',
    'Meta-Group 8': 'ClickUp Workflows & Integrations',
    'Meta-Group 9': 'Login & Access Issues',
    'Meta-Group 10': 'Security Software Integration & Setup',
    'Meta-Group 11': 'Digital Marketing Strategy & Performance',
    'Meta-Group 12': 'Collaboration Tools Connectivity & Hardware Issues',
    'Meta-Group 13': 'Email & Marketing Platform Integrations',
    'Meta-Group 14': 'Data & Analytics Platform Integrations Issues',
    'Meta-Group 15': 'IntelliJ IDEA Issues & Upgrades',
    'Meta-Group 16': 'Hardware Issues & Customer Support',
    'Meta-Group 17': 'AI & Analytics Tool Integrations',
    'Meta-Group 18': 'Jira Configuration & Support',
    'Meta-Group 19': 'General Support Requests',
    'Meta-Group 20': 'Redis Integration & Analytics',
    'Meta-Group 21': 'Network Connectivity & Router Issues',
    'Meta-Group 22': 'Zapier Integration & Workflow Automation',
    'Meta-Group 23': 'Printer Hardware & Connectivity Issues',
    'Meta-Group 24': 'Service & Subscription Information Requests',
    'Meta-Group 25': 'Data Security & Breach Mitigation',
    'Meta-Group 26': 'Consumer Hardware Issues & Purchase Support',
    'Meta-Group 27': 'Application & Software Stability Issues',
    'Meta-Group 28': 'Elasticsearch Integration & Analytics Optimization',
    'Meta-Group 29': 'Cloud & SaaS Integration Issues'
}

In [35]:
final_cluster_names = {
    'Meta-Group 0': 'Financial Data Analytics Support',
    'Meta-Group 1': 'Service Downtime & System Outages',
    'Meta-Group 2': 'Application & System Issues',
    'Meta-Group 3': 'Product & Documentation Inquiries',
    'Meta-Group 4': 'Billing & Payment Issues',
    'Meta-Group 5': 'Third-Party Tool Integrations',
    'Meta-Group 6': 'Service Downtime & System Outages',       # duplicate of 1
    'Meta-Group 7': 'Application Functionality & Sync Issues',
    'Meta-Group 8': 'Software & Platform Support',
    'Meta-Group 9': 'Login & Access Issues',
    'Meta-Group 10': 'Security Software Integration & Setup',
    'Meta-Group 11': 'Digital Marketing Strategy & Performance',
    'Meta-Group 12': 'Collaboration Tools Connectivity & Hardware Issues',
    'Meta-Group 13': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 14': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 15': 'Software & Platform Support',            # merged into 8
    'Meta-Group 16': 'Consumer Hardware Issues & Purchase Support',
    'Meta-Group 17': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 18': 'Software & Platform Support',            # merged into 8
    'Meta-Group 19': 'General Support Requests',
    'Meta-Group 20': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 21': 'Network Connectivity & Router Issues',
    'Meta-Group 22': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 23': 'Consumer Hardware Issues & Purchase Support', # merged into 16
    'Meta-Group 24': 'Billing & Payment Issues',               # merged into 4
    'Meta-Group 25': 'Security Software Integration & Setup',  # merged into 10
    'Meta-Group 26': 'Consumer Hardware Issues & Purchase Support', # merged into 16
    'Meta-Group 27': 'Application & System Issues',            # merged into 2
    'Meta-Group 28': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 29': 'Third-Party Tool Integrations'           # merged into 5
}

In [36]:
df["merged_category"] = df["meta_label_name"].map(final_cluster_names)

df["merged_category"] = df["merged_category"].fillna("Uncategorized / Rare Issues")

print(df["merged_category"].value_counts())

merged_category
Third-Party Tool Integrations                         9523
Financial Data Analytics Support                      5182
Security Software Integration & Setup                 3197
Software & Platform Support                           2227
Service Downtime & System Outages                     1788
Application & System Issues                           1614
Consumer Hardware Issues & Purchase Support            339
Login & Access Issues                                  301
Application Functionality & Sync Issues                169
Collaboration Tools Connectivity & Hardware Issues     162
Digital Marketing Strategy & Performance               160
General Support Requests                               125
Billing & Payment Issues                               118
Product & Documentation Inquiries                       98
Network Connectivity & Router Issues                    44
Name: count, dtype: int64


In [37]:
import json
with open("merged_cluster_names.json", "w") as f:
    json.dump(final_cluster_names, f, indent=2)

In [38]:
import pickle

with open('umap_model.pkl', 'wb') as f:
    pickle.dump(umap_model, f)

with open('knn_model.pkl', 'wb') as f:
    pickle.dump(knn, f)

with open('category_mapping.pkl', 'wb') as f:
    pickle.dump(final_cluster_names, f)

In [39]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = " ".join(text.split())
    return text

def classify_new_ticket(ticket_text):
    clean_text = preprocess_text(ticket_text)

    new_embedding = embedder.encode([clean_text], normalize_embeddings=True)

    raw_pred = knn.predict(new_embedding)
    cluster_id = int(raw_pred[0])

    meta_id = cluster_to_meta_map.get(cluster_id, -1)
    key = f"Meta-Group {meta_id}"
    category = final_cluster_names.get(key, "Uncategorized")

    return category

In [40]:
clean = "Our latest campaign on Facebook is underperforming and the CPC is way too high. We need to re-evaluate the targeting."
print(f"Clean Result: {classify_new_ticket(clean)}")

Clean Result: Security Software Integration & Setup


In [41]:
import joblib

In [ ]:
joblib.dump(embeddings, 'best_embedding.joblib')

['best_embedding.joblib']

In [42]:
embedder.save('/content/embedder')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [44]:
import shutil
from google.colab import files

In [45]:
folder_to_zip = "/content/embedder"

output_zip = "/content/best_embedder"

shutil.make_archive(output_zip, 'zip', folder_to_zip)

'/content/best_embedder.zip'